# Colab setup (run top to bottom)
1. Runtime > Change runtime type > T4 GPU.
2. Change YOUR_NAME below, then run each cell in order.
3. You will be asked once to paste your Kaggle API token (never saved in this notebook).
4. Data is downloaded from Kaggle into the Colab VM (not git). It is wiped when the session ends; just re-run.

In [ ]:
import os
from getpass import getpass

os.environ['RUN_USER'] = 'Kevin Yang'

!rm -rf /content/repo
!git clone -q -b Kevin_Experiment https://github.com/JerNB/daml-indoor67.git /content/repo

%cd /content/repo

!pip -q install timm kaggle

import torch
print('GPU available:', torch.cuda.is_available())

!git branch --show-current

/content/repo
GPU available: True


In [ ]:
# 2) data: download from Kaggle (your account must have joined the competition and accepted the rules)
os.environ['KAGGLE_API_TOKEN'] = getpass('Kaggle API token: ')
!mkdir -p /content/dl /content/data
!kaggle competitions download -c daml-aitp-fall-26-challenge -p /content/dl
!cd /content/dl && unzip -q -o '*.zip' -d /content/data && ls /content/data

Kaggle API token: ··········
100% 272M/272M [00:02<00:00, 108MB/s] 

classes.csv  images  sample_submission.csv  test.csv  train.csv


In [ ]:
# 3) locate the folder that holds train.csv (handles an extra nesting level) and verify layout
from pathlib import Path
hits = list(Path('/content/data').rglob('train.csv'))
assert hits, 'train.csv not found - check the download cell output'
root = hits[0].parent
os.environ['DATA_ROOT'] = str(root)
for need in ['classes.csv', 'sample_submission.csv', 'images/train', 'images/test']:
    print(need, 'OK' if (root / need).exists() else 'MISSING')
print('DATA_ROOT =', root)

classes.csv OK
sample_submission.csv OK
images/train OK
images/test OK
DATA_ROOT = /content/data


In [ ]:
# 4) smoke test: baseline end to end (a few minutes on a GPU)
!python src/extract_features.py --model resnet50.a1_in1k
!python src/linear_probe.py --model resnet50.a1_in1k --name baseline
!python src/summarize.py


model.safetensors: downloading bytes:   0% 0.00/102M [00:00<?, ?B/s]
model.safetensors: downloading bytes:  15% 15.3M/102M [00:00<00:03, 23.0MB/s,  774kB/s  ]
model.safetensors: downloading bytes:  25% 25.2M/102M [00:01<00:01, 39.1MB/s, 1.49MB/s  ]
model.safetensors: downloading bytes:  33% 34.3M/102M [00:01<00:01, 41.4MB/s, 2.91MB/s  ]
model.safetensors: downloading bytes:  46% 47.4M/102M [00:01<00:01, 51.0MB/s, 3.86MB/s  ]
model.safetensors: downloading bytes:  57% 58.3M/102M [00:01<00:00, 52.2MB/s, 4.89MB/s  ]
model.safetensors: downloading bytes:  78% 79.8M/102M [00:02<00:00, 54.1MB/s, 6.56MB/s  ]
model.safetensors: downloading bytes:  87% 88.7M/102M [00:02<00:00, 50.8MB/s, 7.60MB/s  ]
model.safetensors: downloading bytes:  93% 95.5M/102M [00:02<00:00, 54.0MB/s, 7.93MB/s  ]
model.safetensors: downloading bytes: 100% 96.2M/96.2M [00:02<00:00, 37.8MB/s, 8.48MB/s  ]
model.safetensors: reconstructing file: 100% 102M/102M [00:02<00:00, 40.3MB/s, 9.20MB/s  ]
train: 100% 84/84 [00:27<00: